# Netflix Movies and TV Shows — Data Check

This notebook loads the project dataset without third-party dependencies, summarizes its contents, and checks common data-quality problems.

In [ ]:
from collections import Counter
from pathlib import Path
import csv

# This works when Jupyter starts in either the repository root or its notebooks folder.
candidates = [Path('data/Netflix_Movies_and_TV_Shows.csv'), Path('../data/Netflix_Movies_and_TV_Shows.csv')]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find data/Netflix_Movies_and_TV_Shows.csv. Start Jupyter from the project folder.')

with data_path.open(encoding='utf-8-sig', newline='') as file:
    rows = list(csv.DictReader(file))

print(f'Loaded {len(rows):,} rows from {data_path.resolve()}')
print('Columns:', ', '.join(rows[0]) if rows else 'No columns found')

In [ ]:
# Preview records while masking the card-number field.
safe_columns = [column for column in rows[0] if column != 'Card Number'] if rows else []
for row in rows[:5]:
    print({column: row[column] for column in safe_columns})

In [ ]:
def counts_for(column):
    return Counter(row[column].strip() or '<missing>' for row in rows)

for column in ('Type', 'Genre', 'Rating', 'Country'):
    print(f'\n{column}:')
    for value, count in counts_for(column).most_common(10):
        print(f'  {value:<20} {count:>4}')

In [ ]:
# Validate fields and relationships between fields.
missing_by_column = {
    column: sum(not row[column].strip() for row in rows)
    for column in (rows[0] if rows else [])
}
movie_with_seasons = [row for row in rows if row['Type'] == 'Movie' and 'Season' in row['Duration']]
show_with_minutes = [row for row in rows if row['Type'] == 'TV Show' and 'min' in row['Duration']]
invalid_years = []
for row in rows:
    try:
        year = int(row['Release Year'])
        if not 1900 <= year <= 2100:
            invalid_years.append(row)
    except ValueError:
        invalid_years.append(row)

print('Missing values:', {key: value for key, value in missing_by_column.items() if value})
print(f'Movies measured in seasons: {len(movie_with_seasons):,}')
print(f'TV shows measured in minutes: {len(show_with_minutes):,}')
print(f'Invalid release years: {len(invalid_years):,}')
print(f'Duplicate full rows: {len(rows) - len({tuple(row.items()) for row in rows}):,}')

## Interpretation

A movie should normally have a duration in minutes, while a TV show should normally have a duration in seasons. Rows that violate that relationship need correction or exclusion before duration-based analysis. The `Card Number` column is unrelated to title analysis and may represent sensitive or synthetic data, so it is intentionally excluded from previews and should be removed unless there is a documented analytical purpose for it.